In [ ]:
import sys
sys.path.insert(0, "/home/kkingstoun/git/containers_admin2/compute-lib/mmpp")
print("mmpp path added:", sys.path[0])


# Vortex Dynamics Analysis - full numerical workflow

This notebook runs a full vortex post-processing workflow on a real simulation:

`/mnt/storage_6/project_data/pl0095-01/mateuszz/microlab/projects/marie_cuire_vortex_stt/workspace/scratch/minimalmodel/v1/gptpro_fast.zarr`

Pipeline covered:
1. Data loading and metadata inspection
2. Core tracking (maximum / centroid / gaussian)
3. Topology detection
4. Trajectory analysis (orbit, phase, directional spectrum)
5. Spectrum + mode classification
6. Events, signals, and energy post-processing
7. Nonlinear metrics + numerical to analytical bridge
8. Thiele adapters with auto-inferred parameters


## 1) Setup


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zarr

from IPython.display import display

import mmpp

print("mmpp version:", mmpp.__version__)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 4)


def safe_tight_layout(fig=None):
    try:
        (fig or plt.gcf()).tight_layout()
    except RuntimeError:
        pass


## 2) Load simulation


In [ ]:
ZARR_PATH = "/mnt/storage_6/project_data/pl0095-01/mateuszz/microlab/projects/marie_cuire_vortex_stt/workspace/scratch/minimalmodel/v1/gptpro_fast.zarr"

scan = mmpp.MMPP(ZARR_PATH)
if len(scan) == 0:
    raise RuntimeError(f"No jobs found in: {ZARR_PATH}")

sim = scan[0]
z_root = zarr.open(ZARR_PATH, mode="r")

display(scan)
display(sim)

print("Top-level keys:", list(z_root.keys()))
if "m" in z_root:
    print("m shape:", z_root["m"].shape, "dtype:", z_root["m"].dtype)
if "table" in z_root:
    print("table columns:", len(list(z_root["table"].keys())))


In [ ]:
attrs = dict(sim.attrs)
attr_preview = pd.DataFrame([
    {"key": k, "value": str(v)} for k, v in sorted(attrs.items())
])
display(attr_preview.head(40))

if "table" in z_root:
    table = z_root["table"]
    table_info = []
    for key in table.keys():
        arr = table[key]
        table_info.append({"column": key, "shape": tuple(arr.shape), "dtype": str(arr.dtype)})
    display(pd.DataFrame(table_info).sort_values("column").reset_index(drop=True))


def find_column(table_group, aliases):
    lower_map = {str(k).lower(): str(k) for k in table_group.keys()}
    for alias in aliases:
        key = lower_map.get(str(alias).lower())
        if key is not None:
            return key
    return None


## 3) Build vortex interface


In [ ]:
data = sim.m
vortex = data.vortex

display(vortex)

print("dataset shape:", data.shape)
print("dataset dt [s]:", getattr(data, "dt", "n/a"))
print("dx, dy [m]:", attrs.get("dx"), attrs.get("dy"))


## 4) Core tracking (maximum / centroid / gaussian)


In [ ]:
tracking = {}
errors = {}
for method in ("maximum", "centroid", "gaussian"):
    try:
        tr = vortex.track(method=method, z_layer=0, force=True)
        tracking[method] = tr
    except Exception as exc:
        errors[method] = repr(exc)

if not tracking:
    raise RuntimeError(f"Tracking failed for all methods: {errors}")

traj = tracking.get("gaussian") or next(iter(tracking.values()))
print("selected trajectory method:", traj.method)

rows = []
for method, tr in tracking.items():
    mean_radius_nm = float(np.mean(tr.r) * 1e9) if tr.time.size else float("nan")
    mean_freq_ghz = float(np.mean(tr.instantaneous_frequency) / (2.0 * np.pi * 1e9)) if tr.time.size > 1 else float("nan")
    rows.append(
        {
            "method": method,
            "n_samples": int(tr.time.size),
            "rotation": tr.rotation_sense,
            "mean_radius_nm": mean_radius_nm,
            "mean_freq_ghz": mean_freq_ghz,
            "confidence_mean": float(np.mean(tr.confidence)) if tr.confidence.size else float("nan"),
        }
    )

display(pd.DataFrame(rows).sort_values("method").reset_index(drop=True))

if errors:
    print("tracking errors:", errors)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
traj.plt.xy(ax=axes[0])
axes[0].set_title("Tracked core: x(t), y(t)")
traj.plt.orbit_2d(ax=axes[1])
axes[1].set_title("Tracked orbit")
safe_tight_layout()
plt.show()


## 5) Optional check against table core positions (`ext_coreposx/y`)


In [ ]:
if "table" in z_root:
    table = z_root["table"]
    key_x = find_column(table, ("ext_coreposx", "VortexX", "vortexx", "coreposx"))
    key_y = find_column(table, ("ext_coreposy", "VortexY", "vortexy", "coreposy"))

    if key_x is not None and key_y is not None:
        x_ref = np.asarray(table[key_x][:], dtype=float)
        y_ref = np.asarray(table[key_y][:], dtype=float)

        n = int(min(traj.time.size, x_ref.size, y_ref.size))
        x_num = np.asarray(traj.x[:n], dtype=float)
        y_num = np.asarray(traj.y[:n], dtype=float)
        x_ref = x_ref[:n]
        y_ref = y_ref[:n]

        rmse_nm = float(np.sqrt(np.mean((x_num - x_ref) ** 2 + (y_num - y_ref) ** 2)) * 1e9)
        print(f"reference columns: {key_x}, {key_y}")
        print(f"alignment length : {n}")
        print(f"RMSE(num vs table): {rmse_nm:.4f} nm")

        fig, ax = plt.subplots(figsize=(5.6, 5.0))
        ax.plot(x_num * 1e9, y_num * 1e9, label="numerical track", lw=1.0)
        ax.plot(x_ref * 1e9, y_ref * 1e9, label="table core pos", lw=1.0, ls="--")
        ax.set_xlabel("x [nm]")
        ax.set_ylabel("y [nm]")
        ax.set_title("Orbit overlay: tracked vs table")
        ax.set_aspect("equal")
        ax.grid(True, alpha=0.25)
        ax.legend()
        safe_tight_layout()
        plt.show()
    else:
        print("No ext_coreposx/y (or VortexX/Y) columns in table.")


## 6) Topology snapshots


In [ ]:
frame_ids = sorted(set([0, int(traj.time.size // 2), int(max(traj.time.size - 1, 0))]))

topo_rows = []
topo_by_frame = {}
for fid in frame_ids:
    topo = vortex.topology.detect(t=int(fid), method="finite_diff", z_layer=0, force=True)
    topo_by_frame[int(fid)] = topo
    topo_rows.append(
        {
            "frame": int(fid),
            "state": topo.state,
            "polarity": int(topo.polarity),
            "vorticity": int(topo.vorticity),
            "chirality": int(topo.chirality),
            "Q": float(topo.Q),
            "confidence": float(topo.confidence),
            "is_consistent": bool(topo.is_consistent),
        }
    )

display(pd.DataFrame(topo_rows))

mid = topo_by_frame[frame_ids[len(frame_ids) // 2]]
q_map = getattr(mid, "topological_density", None)
if q_map is not None and np.size(q_map):
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    im = ax.imshow(np.asarray(q_map, dtype=float), origin="lower", cmap="coolwarm")
    ax.set_title("Topological density (middle frame)")
    plt.colorbar(im, ax=ax, label="q")
    safe_tight_layout()
    plt.show()


## 7) Trajectory analysis (orbit, phase, directional spectrum)


In [ ]:
orbit_fit = traj.analysis.orbit.fit(model="ellipse")
phase_wrapped = traj.analysis.phase.instantaneous(method="complex")
phase_unwrapped = np.unwrap(phase_wrapped)
freq_ghz = traj.analysis.phase.frequency(method="complex", unit="ghz")
directional = traj.analysis.spectrum.directional(method="welch")

print(f"orbit center [nm]: ({orbit_fit.center[0] * 1e9:.4f}, {orbit_fit.center[1] * 1e9:.4f})")
print(f"orbit radius [nm]: {orbit_fit.radius * 1e9:.4f}")
print(f"orbit eccentricity: {orbit_fit.eccentricity:.4f}")
print(f"phase-based median f [GHz]: {np.nanmedian(freq_ghz):.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].plot(traj.time * 1e9, phase_unwrapped / (2.0 * np.pi), lw=1.0)
axes[0].set_xlabel("t [ns]")
axes[0].set_ylabel("phase / 2pi [cycles]")
axes[0].set_title("Unwrapped phase")
axes[0].grid(True, alpha=0.25)

axes[1].plot(traj.time * 1e9, freq_ghz, lw=1.0)
axes[1].set_xlabel("t [ns]")
axes[1].set_ylabel("f_inst [GHz]")
axes[1].set_title("Instantaneous frequency")
axes[1].grid(True, alpha=0.25)

directional.plt.power_spectrum(ax=axes[2], unit="ghz")
axes[2].set_title("Directional spectrum (CCW/CW)")

safe_tight_layout()
plt.show()


## 8) Spectrum namespace + mode classification


In [ ]:
gyr = vortex.spectrum.gyration(method="welch")
breath = vortex.spectrum.breathing(method="welch")
sgram = vortex.spectrum.spectrogram(component="radius")

print(f"gyration peak [GHz]: {gyr.peak_frequency_ghz:.4f}")
print(f"breathing peak [GHz]: {breath.peak_frequency_ghz:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
gyr.plt.power_spectrum(ax=axes[0], as_ghz=True, log_scale=True)
axes[0].set_title("Gyration PSD")

breath.plt.power_spectrum(ax=axes[1], as_ghz=True, log_scale=True)
axes[1].set_title("Breathing PSD")

sgram.plt.spectrogram(ax=axes[2], as_ghz=True, db_scale=True)
axes[2].set_title("Radius spectrogram")

safe_tight_layout()
plt.show()

modes = vortex.modes.classify_all(max_modes=8, min_prominence=0.03)
mode_table = vortex.modes.plt.mode_table()
print("detected modes:", len(modes))
if mode_table:
    display(pd.DataFrame(mode_table))

fig, ax = plt.subplots(figsize=(8.0, 4.0))
vortex.modes.plt.mode_map(ax=ax)
safe_tight_layout()
plt.show()


## 9) Event detection


In [ ]:
pol_switch = vortex.events.polarity_switches(trajectory=traj)
state_switch = vortex.events.state_switches(trajectory=traj)
expulsion = vortex.events.core_expulsions(trajectory=traj)
dwell_g = vortex.events.dwell_times(trajectory=traj, state="G-state")

print("polarity switches:", len(pol_switch))
print("state switches   :", len(state_switch))
print("core expulsions  :", len(expulsion))
print("G-state dwells   :", dwell_g.count)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
vortex.events.plt.event_timeline(trajectory=traj, ax=axes[0])
axes[0].set_title("Event timeline")

if dwell_g.count > 0:
    dwell_g.plt.dwell_histogram(ax=axes[1])
else:
    axes[1].text(0.5, 0.5, "No dwell intervals detected", ha="center", va="center")
    axes[1].set_axis_off()

safe_tight_layout()
plt.show()


## 10) Signals and energy post-processing


In [ ]:
mr = vortex.signals.magnetoresistance(trajectory=traj)
voltage = vortex.signals.voltage(trajectory=traj, magnetoresistance=mr)
signal_psd = vortex.signals.power_spectrum(signal="voltage", trajectory=traj)

energy_ts = vortex.energy.time_resolved(force=True)
potential = vortex.energy.potential(trajectory=traj, method="auto", force=True)
pinning = vortex.energy.pinning(potential=potential, force=True)

print("signals method:", mr.method)
print("MR mean [Ohm]:", mr.mean_resistance_ohm)
print("V_rms [V]:", voltage.rms_voltage_v)
print("signal peak [GHz]:", signal_psd.peak_frequency_ghz)
print("energy channels:", energy_ts.available_channels)
print("pinning sites:", len(pinning.sites), "(method:", potential.method + ")")

fig, axes = plt.subplots(2, 3, figsize=(16, 8.5))
mr.plt.time_trace(ax=axes[0, 0])
axes[0, 0].set_title("Magnetoresistance")

voltage.plt.time_trace(ax=axes[0, 1])
axes[0, 1].set_title("Voltage")

signal_psd.plt.power_spectrum(ax=axes[0, 2])
axes[0, 2].set_title("Voltage PSD")

if energy_ts.available_channels:
    energy_ts.plt.time_resolved(ax=axes[1, 0])
else:
    axes[1, 0].text(0.5, 0.5, "No energy channels found", ha="center", va="center")
    axes[1, 0].set_axis_off()

potential.plt.potential(ax=axes[1, 1], as_nev=True)
axes[1, 1].set_title("Effective potential")

pinning.plt.potential_with_sites(ax=axes[1, 2], as_nev=True)
axes[1, 2].set_title("Pinning minima")

safe_tight_layout()
plt.show()


## 11) Nonlinear analysis + bridge fit + Thiele adapters


In [ ]:
st = vortex.nonlinear.slavin_tiberkevich(trajectory=traj)
force_balance = vortex.nonlinear.force_balance(trajectory=traj)
bridge_fit = vortex.bridge.fit.thiele_from_trajectory(traj, damping=0.01)
cmp_proxy = traj.compare.with_(bridge_fit.simulated_trajectory, label=("numerical", "thiele_proxy"))

print(f"ST f0 [GHz]: {st.f_0_ghz:.4f}")
print(f"ST N [rad/s]: {st.N:.4e}")
print(f"ST linewidth [MHz]: {st.linewidth_hz * 1e-6:.4f}")
print("linewidth_resolution_limited:", st.linewidth_resolution_limited)
print(f"bridge delta_f_mean [Hz]: {cmp_proxy.metrics.delta_f_mean:.4e}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
cmp_proxy.plot.overlay_orbit(ax=axes[0])
axes[0].set_title("Numerical vs Thiele proxy (bridge fit)")

force_balance.plt.force_balance(ax=axes[1], as_norm=True)
axes[1].set_title("Thiele force balance")

safe_tight_layout()
plt.show()

cpp_adapter = vortex.model.thiele.cpp()
cip_adapter = vortex.model.thiele.cip()

model_rows = [
    {"parameter": "Ms [A/m]", "value": cpp_adapter.model.material.Ms, "source": "attrs: Ms/Msat"},
    {"parameter": "alpha", "value": cpp_adapter.model.material.alpha, "source": "attrs: alpha"},
    {"parameter": "P", "value": cpp_adapter.model.material.P, "source": "attrs: P/Pol"},
    {"parameter": "A [J/m]", "value": cpp_adapter.model.material.A, "source": "attrs: Aex/A"},
    {"parameter": "R [m]", "value": cpp_adapter.model.geom.R, "source": "dataset shape + dx/dy"},
    {"parameter": "L [m]", "value": cpp_adapter.model.geom.L, "source": "attrs: thickness/L/dz*Nz"},
    {"parameter": "polarity", "value": cpp_adapter.model.polarity, "source": "attrs: polarity/p"},
    {"parameter": "omega0 [rad/s]", "value": cpp_adapter.model.omega0, "source": "omega0_novosad(material, geom)"},
]
display(pd.DataFrame(model_rows))

if traj.time.size > 1:
    dt = float(np.median(np.diff(traj.time)))
else:
    dt = 1e-12
k = max(int(traj.time.size) - 1, 1)
t_end = float(np.nextafter(np.float64(dt) * np.float64(k), np.inf))

try:
    traj_cpp = cpp_adapter.simulate(t_span=(0.0, t_end), dt=dt, J_func="auto_from_table")
    traj_cip = cip_adapter.simulate(t_span=(0.0, t_end), dt=dt, J_func="auto_from_table")

    cmp_cpp = traj.compare.with_(traj_cpp, label=("numerical", "thiele_cpp"))
    cmp_cip = traj.compare.with_(traj_cip, label=("numerical", "thiele_cip"))

    print(f"delta_f_mean num-vs-cpp [Hz]: {cmp_cpp.metrics.delta_f_mean:.4e}")
    print(f"delta_f_mean num-vs-cip [Hz]: {cmp_cip.metrics.delta_f_mean:.4e}")

    fig, ax = plt.subplots(figsize=(6.0, 5.0))
    cmp_cpp.plot.overlay_orbit(ax=ax)
    ax.plot(traj_cip.x, traj_cip.y, ls=":", lw=1.1, label="thiele_cip")
    ax.legend()
    ax.set_title("Numerical vs analytical adapters")
    safe_tight_layout()
    plt.show()
except Exception as exc:
    print("Analytical adapter simulation failed:", repr(exc))


## 12) Summary and practical notes

Short answer to: "Does the library auto-read simulation parameters for analytical models?"

- Yes, partially.
- The Thiele adapters auto-infer key parameters from simulation metadata and dataset geometry.
- Dynamic fitting quality still depends on manual calibration and/or fit strategy.


In [ ]:
summary = {
    "zarr_path": ZARR_PATH,
    "n_time_samples": int(traj.time.size),
    "tracking_method": traj.method,
    "rotation_sense": traj.rotation_sense,
    "gyration_peak_ghz": float(gyr.peak_frequency_ghz),
    "breathing_peak_ghz": float(breath.peak_frequency_ghz),
    "topology_Q_mid": float(topo_by_frame[frame_ids[len(frame_ids) // 2]].Q),
    "st_f0_ghz": float(st.f_0_ghz),
    "st_linewidth_mhz": float(st.linewidth_hz * 1e-6),
    "bridge_delta_f_hz": float(cmp_proxy.metrics.delta_f_mean),
    "energy_channels": ", ".join(energy_ts.available_channels) if energy_ts.available_channels else "none",
}

display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

print("\nAuto-inference coverage in this state:")
print("- material params (Ms, alpha, P, A): YES (from attrs, with defaults)")
print("- geometry (R, L): YES (from dataset shape + attrs)")
print("- polarity: YES (from attrs/default)")
print("- current waveform for analytical model: PARTIAL (table J aliases or fallback 0)")
print("- best-fit Thiele dynamics to numerics: PARTIAL (bridge proxy fit is available; full calibrated fit is still task-specific)")
